# Simulate Static and Continuous Batching

Companion notebook for the [Continuous Batching](https://codemia.io/courses/llm_inference_and_serving/llm_continuous_batching) lesson on Codemia.

**What you will build**: a step level simulator of one GPU serving an 8B model. Each decode step costs the time to read the weights plus the KV cache of the batch, from the roofline model, and prefill costs compute per prompt token. You will run the same Poisson traffic through static batching, which waits for a full batch and runs it until its longest member finishes, and through continuous batching, which admits and retires sequences at every step. Then you will plot throughput and latency against load and find the load each scheduler can actually sustain.

No GPU. Standard library, `numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The cost of a step

Decode is memory bound: a step reads all weights once plus every cached token of every sequence in the batch. Prefill is compute bound: about `2 * params` FLOPs per prompt token. These two numbers are enough to reproduce the shape of real engine benchmarks.

In [ ]:
PARAMS = 8e9
WEIGHT_BYTES = 16e9            # 8B in 16 bit
KV_PER_TOKEN = 128 * 1024      # bytes, Llama 3 8B
BW = 3.35e12                   # H100 HBM bytes per second
FLOPS = 600e12                 # realistic dense bf16
OVERHEAD = 0.002               # seconds per step for scheduling and launch
MAX_BATCH = 128

def decode_step_s(cached_tokens):
    return (WEIGHT_BYTES + cached_tokens * KV_PER_TOKEN) / BW + OVERHEAD

def prefill_s(prompt_tokens):
    return 2 * PARAMS * prompt_tokens / FLOPS

for b in [1, 8, 32, 128]:
    t = decode_step_s(b * 1500)
    print("batch %3d at 1,500 tokens each: step %.1f ms, %6.0f tokens/s" % (b, 1000 * t, b / t))

Throughput grows almost linearly with batch size because each step still reads the weights once. That is the whole reason to batch.

## 2. Traffic

Requests arrive as a Poisson process. Prompt and output lengths are lognormal: most are short, some are long. The long outputs are what hurts static batching.

In [ ]:
def make_requests(rate, seconds=120, seed=0):
    rng = np.random.default_rng(seed)
    n = rng.poisson(rate * seconds)
    arrive = np.sort(rng.uniform(0, seconds, n))
    prompt = np.clip(rng.lognormal(np.log(800), 0.8, n), 16, 8000).astype(int)
    output = np.clip(rng.lognormal(np.log(200), 1.0, n), 4, 4000).astype(int)
    return arrive, prompt, output

a, p, o = make_requests(5)
print("%d requests, median output %d tokens, p99 output %d tokens" % (len(a), np.median(o), np.percentile(o, 99)))

## 3. Two schedulers

**Static**: wait until `MAX_BATCH` requests are queued or the oldest has waited `max_wait`, prefill them together, then decode until every member is done. Finished members sit idle in the batch.

**Continuous**: at every step, retire finished sequences, admit waiting ones up to `MAX_BATCH`, prefill the newcomers in that step, then decode one token for everyone.

Both return per request TTFT and end to end latency plus the overall output token throughput.

In [ ]:
def run_static(reqs, max_wait=0.5):
    arrive, prompt, output = reqs
    n = len(arrive); t = 0.0; i = 0
    ttft = np.zeros(n); e2e = np.zeros(n)
    while i < n:
        t = max(t, arrive[i])
        deadline = arrive[i] + max_wait
        j = i
        while j < n and j - i < MAX_BATCH and arrive[j] <= max(t, deadline):
            j += 1
        t = max(t, min(deadline, arrive[j - 1]) if j - i < MAX_BATCH else arrive[j - 1])
        idx = np.arange(i, j)
        t += prefill_s(prompt[idx].sum())
        ttft[idx] = t - arrive[idx]
        ctx = prompt[idx].astype(float)
        steps = output[idx].max()
        for s in range(1, steps):
            t += decode_step_s(ctx.sum())           # finished members still occupy the batch
            ctx += 1
            done = output[idx] == s + 1
            e2e[idx[done]] = t - arrive[idx[done]]
        one = output[idx] == 1
        e2e[idx[one]] = ttft[idx[one]]
        i = j
    return ttft, e2e, output.sum() / t

def run_continuous(reqs):
    arrive, prompt, output = reqs
    n = len(arrive); t = 0.0; nxt = 0
    ttft = np.zeros(n); e2e = np.zeros(n)
    active = []                                     # [request, tokens produced, context]
    while nxt < n or active:
        if not active and nxt < n:
            t = max(t, arrive[nxt])
        new = []
        while nxt < n and arrive[nxt] <= t and len(active) + len(new) < MAX_BATCH:
            new.append(nxt); nxt += 1
        step = decode_step_s(sum(c for _, _, c in active)) if active else 0.0
        if new:
            step += prefill_s(prompt[new].sum())
        t += step
        for r in new:
            ttft[r] = t - arrive[r]
            active.append([r, 1, prompt[r] + 1])
        keep = []
        for item in active:
            r, k, c = item
            if r not in new:
                item[1] += 1; item[2] += 1
            if item[1] >= output[r]:
                e2e[r] = t - arrive[r]
            else:
                keep.append(item)
        active = keep
    return ttft, e2e, output.sum() / t

reqs = make_requests(5)
for name, fn in [("static", run_static), ("continuous", run_continuous)]:
    ttft, e2e, tput = fn(reqs)
    print("%-10s TTFT p50 %6.2f s p99 %6.2f s | e2e p50 %6.2f s | %6.0f output tok/s" %
          (name, np.median(ttft), np.percentile(ttft, 99), np.median(e2e), tput))

## 4. Throughput and latency under load

Sweep the arrival rate. Throughput rises with load until the scheduler saturates, then the queue grows without bound and latency explodes. The knee of each curve is the sustainable load.

In [ ]:
rates = [1, 2, 4, 6, 8, 12, 16, 20]
res = {"static": [], "continuous": []}
for r in rates:
    reqs = make_requests(r, seconds=60, seed=r)
    for name, fn in [("static", run_static), ("continuous", run_continuous)]:
        ttft, e2e, tput = fn(reqs)
        res[name].append((tput, np.percentile(ttft, 99), np.percentile(e2e, 50)))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, rows in res.items():
    ax[0].plot(rates, [x[0] for x in rows], marker="o", label=name)
    ax[1].plot(rates, [x[1] for x in rows], marker="o", label=name)
ax[0].set_xlabel("arrivals per second"); ax[0].set_ylabel("output tokens per second"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set_xlabel("arrivals per second"); ax[1].set_ylabel("p99 TTFT (s)"); ax[1].set_yscale("log"); ax[1].legend(); ax[1].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

Static batching wastes steps on finished sequences and makes new arrivals wait for the whole batch, so its TTFT rises long before the GPU is busy. Continuous batching keeps the batch full of live work, which is why every modern engine does it.

## 5. Exercises

1. Lower `MAX_BATCH` to 32. Where does each knee move, and why does the continuous scheduler lose more?
2. Add **chunked prefill**: cap the prompt tokens processed per step at 512 and carry the rest to later steps. Measure what it does to the TPOT of running sequences when a 6,000 token prompt arrives.
3. Give the continuous scheduler a KV cache limit of 400,000 tokens and preempt the newest sequence when it is exceeded. How often does preemption happen at 16 arrivals per second?